In [ ]:
from glob import glob
import os
import pandas as pd
import json
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import shutil
import base64
import numpy as np

import cv2
from ultralytics import YOLO

from openai import OpenAI
import os


client = OpenAI(api_key=KEY, base_url="https://openrouter.ai/api/v1")
MODEL = "mistralai/mistral-small-2603"
MODEL_NAME = "Mistral Small Prompt 1"

In [3]:
# qwen/qwen3.7-plus

# mistralai/mistral-small-2603
# mistralai/mistral-medium-3.1
# mistralai/mistral-large-2512

# openai/gpt-5-mini
# openai/gpt-4.1

# anthropic/claude-opus-4.8
# anthropic/claude-sonnet-4.6

In [49]:
def summarize_letter(
    client: OpenAI,
    transcription: str,
    model: str = MODEL,
) -> str:
    """
    Summarize a transcribed historical French letter using a Mistral model via OpenRouter.

    Args:
        client: OpenAI-compatible client pointed at OpenRouter
        transcription: The transcribed text of the letter as a string
        model: OpenRouter model string

    Returns:
        A short description of the letter's content
    """

    system_prompt = """
Tu es un expert en correspondance française des XIXe et XXe siècles.
On te fournit la transcription d'un document écrit, souvent un élément de correspondances, issu des archives personnelles de Jean-Louis Forain (1852–1931), peintre et caricaturiste français.

Ton rôle est de produire une courte description en français (1 à 2 phrases) résumant le contenu de la lettre.

Règles :
- Si l'auteur, le destinataire, le lieu ou le contexte sont identifiables, mentionne-les.
- Si ces informations ne sont pas claires, résume simplement ce qui est dit ou exprimé.
- Les noms propres (personnes, lieux, domaines) doivent apparaître tels quels.
- Retourne uniquement la description, sans explication, sans balises markdown.
"""

    user_prompt = f"""Voici la transcription d'un document manuscrit :

{transcription}

Résume en 1 à 2 phrases ce que dit cette lettre."""

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )

    return response.choices[0].message.content

In [28]:
files_path = "/Users/charles-/Downloads/Master/M1/CartaData/Forain/INDUSTRIALISATION/transcriptions/**/*.txt"
output_folder = "/Users/charles-/Downloads/Master/M1/CartaData/Forain/INDUSTRIALISATION/descriptions"
output_folder = os.path.join(output_folder, MODEL_NAME)
os.makedirs(output_folder, exist_ok=True)

records = dict()

for file_path in glob(files_path):
    output_path_elements = file_path.split(sep="/")[-2:]
    subfolder = output_path_elements[0]
    basename = output_path_elements[1].split(sep=".")[0]
    basename = f"{basename}"
    sub_output_folder = os.path.join(output_folder, subfolder)
    output_path = os.path.join(sub_output_folder, f"{basename}.txt")
    os.makedirs(sub_output_folder, exist_ok=True)
    
    try:
        with open(file_path, "r") as input_file:
            transcription = input_file.read()
    except:
        print(f"Error in input: {file_path}")
        continue
    
    summary = summarize_letter(client=client, transcription=transcription, model=MODEL)
    
    try:
        with open(output_path, "w") as output_file:
            output_file.write(summary)
        records[basename] = summary
    except:
        print(f"Error in output: {file_path}")
        print(summary)
    
    records[basename] = summary


df = pd.DataFrame(records)
df.to_csv(os.path.join(output_folder, "summaryOfSummaries.csv"), index=False)

ValueError: If using all scalar values, you must pass an index

In [6]:
print(answer_gpt)

```json
{
  "auteur": "Mme H. Pécaut",
  "destinataire": "Mme R. Pécaut",
  "résumé": "L'auteur décrit une journée agréable à Bayreuth, mentionnant la visite de lieux pittoresques et l'envoi d'une carte postale depuis un café. Elle exprime son désir de continuer à écrire à sa destinataire et évoque des souvenirs d'enfance."
}
```


In [29]:
print(file_path)

./ALLEMAGNE/FO-A-1647_2.tif


## Évaluation des résumés

In [46]:
def evaluate_descriptions_batch(
    client: OpenAI,
    transcription: str,
    descriptions: dict[str, str],  # {model_name: description}
    model: str = MODEL,
) -> dict:
    """
    Evaluate and rank multiple descriptions for the same transcription in one call.

    Args:
        client: OpenAI-compatible client
        transcription: Original transcribed letter text
        descriptions: Dict mapping model name to its generated description
        model: Model string to use for evaluation

    Returns:
        dict with ranking and per-description comments
    """

    descriptions_block = "\n".join(
        f"[{name}]: {desc}" for name, desc in descriptions.items()
    )

    system_prompt = """
Tu es un évaluateur expert en résumé de correspondance historique française.
On te donne la transcription d'une lettre ancienne et plusieurs descriptions générées par différents modèles.

Compare les descriptions entre elles selon 3 critères :
- fidélité : la description reflète-t-elle fidèlement le contenu ? (pas d'hallucination, pas d'omission majeure)
- concision : est-elle brève et sans remplissage ?
- informativité : capture-t-elle les éléments essentiels (qui, quoi, où si disponibles) ?

Retourne uniquement un objet JSON valide, sans commentaire ni balise markdown, de la forme :
{
  "classement": ["modèle_1", "modèle_2", ...],
  "meilleur": "modèle_x",
  "évaluations": {
    "modèle_x": {"fidélité": <1-5>, "concision": <1-5>, "informativité": <1-5>, "commentaire": "..."},
    ...
  }
}

Le classement va du meilleur au moins bon.
"""

    user_prompt = f"""Transcription :
{transcription}

Descriptions à comparer :
{descriptions_block}"""

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )

    raw = response.choices[0].message.content.strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"raw": raw, "error": "JSON parsing failed"}

In [47]:
desc_folder = "/Users/charles-/Downloads/Master/M1/CartaData/Forain/INDUSTRIALISATION/descriptions"
transcription_folder = "/Users/charles-/Downloads/Master/M1/CartaData/Forain/INDUSTRIALISATION/transcriptions"
output_folder = "/Users/charles-/Downloads/Master/M1/CartaData/Forain/INDUSTRIALISATION/LOGS"
    
grouped = dict()  # {basename: {model_name: {"transcription": ..., "description": ...}}}

for model_folder in glob(f"{desc_folder}/**"):
    model_name = model_folder.split("/")[-1]
    for desc_path in glob(f"{model_folder}/**/*.txt"):
        path_elements = desc_path.split(sep="/")[-2:]
        basename = path_elements[1].split(sep=".")[0]
        subfolder = path_elements[0]
        
        try:
            with open(desc_path, "r") as f:
                desc = f.read()
        except:
            print(f"Erreur de lecture desc: {desc_path}")
            continue

        transcription_path = os.path.join(transcription_folder, subfolder, f"{basename}.txt")
        try:
            with open(transcription_path, "r") as f:
                transcription = f.read()
        except:
            print(f"Erreur de lecture transcription: {transcription_path}")
            continue

        if basename not in grouped:
            grouped[basename] = {}
        grouped[basename][model_name] = {"transcription": transcription, "description": desc}

# Evaluate each letter across all its model descriptions
results = dict()

for basename, models in grouped.items():
    # Use the transcription from any model (they should all be the same source)
    transcription = next(iter(models.values()))["transcription"]
    descriptions = {model_name: data["description"] for model_name, data in models.items()}

    results[basename] = evaluate_descriptions_batch(client, transcription, descriptions)

# Save
output_path = os.path.join(output_folder, "evaluations.json")
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)